In [7]:
import csv
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"


def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    with open(path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    print(f"Number of rows found: {len(rows)}")
    print(f"Column names found: {reader.fieldnames}")

    return rows

read_sales(DATA_PATH)
    



Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [43]:

with open(DATA_PATH, mode="r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    sales = list(reader)

headers = sales[0].keys()

# Calculate column widths
widths = {
    header: max(
        len(header),
        max(len(row[header]) for row in sales)
    )
    for header in headers
}

# Print headers
print(" | ".join(header.ljust(widths[header]) for header in headers))

# Print separator
print("-+-".join("-" * widths[header] for header in headers))

# Print rows
for row in sales:
    print(" | ".join(row[header].ljust(widths[header]) for header in headers))

sale_id | sale_date  | store             | product                | quantity | unit_price_kes | payment_method
--------+------------+-------------------+------------------------+----------+----------------+---------------
S-3001  | 2026-09-01 | Westgate          | Safaricom Neon Ray     | 1        | 5999           | Cash          
S-3002  | 2026-09-02 | Westgate          | USB-C Charger 25W      | 2        | 1499           | M-Pesa        
S-3003  | 2026-09-02 | Kisumu Mega Plaza | Lenovo Tab M10         | 2        | 24999          | Cash          
S-3004  | 2026-09-02 | Westgate          | Xiaomi Redmi Buds      | 2        | 3199           | M-Pesa        
S-3005  | 2026-09-03 | Nakuru Westside   | Tecno Spark 20         | 2        | 13499          | M-Pesa        
S-3006  | 2026-09-03 | Nairobi CBD       | Xiaomi Redmi Buds      | two      | 3199           | Cash          
S-3007  | 2026-09-04 | Mombasa City Mall | JBL Go 3 Speaker       | 2        | 5499           | Cash          
S

In [47]:
def parse_number(value: str) -> float | None:
    """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    if value is None:
        return None

    # Clean up whitespace and common non-numeric fluff
    cleaned = value.strip().upper()

    # Drop currency signs and abbreviations (e.g., 'KES', 'KSH', '$')
    cleaned = (
        cleaned.replace("KES", "")
        .replace("KSH", "")
        .replace("SH", "")
        .replace("$", "")
        .strip()
    )

    # Remove commas used as thousands separators (e.g., "1,200.50" -> "1200.50")
    cleaned = cleaned.replace(",", "")

    # Handle explicit placeholders or empty fields
    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    try:
        return float(cleaned)
    except ValueError:
        # String contains untrustworthy characters (e.g., "twelve", "120A")
        return None


print(parse_number("  KES 1,200.50  "))  # Should return 1200.50
print(parse_number("N/A"))  # Should return None

1200.5
None


In [48]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    
    clean_rows: list[dict] = []
    skipped_rows: list[dict] = []

    for row in rows:
        quantity = parse_number(row["quantity"])
        unit_price = parse_number(row["unit_price_kes"])

        reasons = []
        if quantity is None or quantity == 0:
            reasons.append(f"quantity {row['quantity']!r} is missing, zero or not numeric")
        if unit_price is None or unit_price == 0:
            reasons.append(f"unit price {row['unit_price_kes']!r} is missing, zero or not numeric")

        if reasons:
            skipped_rows.append({**row, "reason": "; ".join(reasons)})
            continue

        clean_rows.append({
            **row,
            "quantity": quantity,
            "unit_price_kes": unit_price,
            "revenue": quantity * unit_price,
        })

    return clean_rows, skipped_rows

rows, skipped = convert_rows(raw_rows)
print(f"Clean rows: {len(rows)} | Skipped rows: {len(skipped)}")
for s in skipped:
    print(f"  skipped {s['sale_id']}: {s['reason']}")

Clean rows: 46 | Skipped rows: 4
  skipped S-3006: quantity 'two' is missing, zero or not numeric
  skipped S-3030: unit price 'N/A' is missing, zero or not numeric
  skipped S-3035: quantity '' is missing, zero or not numeric
  skipped S-3039: unit price '' is missing, zero or not numeric


In [24]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate the total revenue from a list of rows."""
    total = 0.0

    for row in rows:
        revenue = row.get("revenue")
        if revenue is not None:
            total += revenue
    return total

total = total_revenue(rows)
print(f"Total Revenue: KES {total:,.2f}")

Total Revenue: KES 1,310,812.00


In [28]:
def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Total revenue grouped by any column (product, store, payment_method...)."""
    totals: dict[str, float] = {}
    for row in rows:
        key = row.get(field)
        revenue = row.get("revenue", 0.0)
        if key is not None:
            totals[key] = totals.get(key, 0.0) + revenue
    return totals

by_product = revenue_by(rows, "product")
by_store = revenue_by(rows, "store")
by_payment_method = revenue_by(rows, "payment_method")

print("Revenue by Product:")
for product, revenue in by_product.items():
    print(f"  {product}: KES {revenue:,.2f}")
print("\nRevenue by Store:")
for store, revenue in by_store.items():
    print(f"  {store}: KES {revenue:,.2f}") 
print("\nRevenue by Payment Method:")
for payment_method, revenue in by_payment_method.items():
    print(f"  {payment_method}: KES {revenue:,.2f}")



Revenue by Product:
  Safaricom Neon Ray: KES 11,998.00
  USB-C Charger 25W: KES 19,487.00
  Lenovo Tab M10: KES 199,992.00
  Xiaomi Redmi Buds: KES 25,592.00
  Tecno Spark 20: KES 161,988.00
  JBL Go 3 Speaker: KES 49,491.00
  Oraimo FreePods: KES 19,593.00
  Anker Power Bank 20000: KES 38,691.00
  Hisense 43in Smart TV: KES 272,993.00
  Samsung Galaxy A15: KES 56,997.00
  HP 15 Laptop: KES 389,994.00
  Infinix Hot 40: KES 63,996.00

Revenue by Store:
  Westgate: KES 235,979.00
  Kisumu Mega Plaza: KES 449,678.00
  Nakuru Westside: KES 139,988.00
  Mombasa City Mall: KES 154,082.00
  Nairobi CBD: KES 331,085.00

Revenue by Payment Method:
  Cash: KES 304,375.00
  M-Pesa: KES 738,568.00
  Card: KES 267,869.00


In [45]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    sorted_totals = sorted(totals.items(), key=lambda item: item[1], reverse=True)
    return sorted_totals[:n]
    
top3 = top_n(by_product, n=3)
print("\nTop 3 Products by Revenue:")
for product, revenue in top3:
    print(f"  {product}: KES {revenue:,.2f}")



Top 3 Products by Revenue:
  HP 15 Laptop: KES 389,994.00
  Hisense 43in Smart TV: KES 272,993.00
  Lenovo Tab M10: KES 199,992.00


In [55]:
THRESHOLD = 50_000


def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Transactions whose revenue is at or above threshold (use a comprehension)."""
    return [row for row in rows if row["revenue"] >= threshold]


big = high_value(rows, THRESHOLD)
print(f"\nHigh-value sales (KES {THRESHOLD:,.2f} or more): {len(big)}")
for r in big:
    print(f"  {r['sale_id']}: {r['product']} at {r['store']} for KES {r['revenue']:,.2f}")

# Count of high-value sales per store (dict comprehension)
stores = {r["store"] for r in rows}
big_by_store = {s: sum(1 for r in big if r["store"] == s) for s in stores}

print("\nHigh-value sales per store:")
for name, count in sorted(big_by_store.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {name:<24} {count}")


High-value sales (KES 50,000.00 or more): 5
  S-3013: Lenovo Tab M10 at Westgate for KES 99,996.00
  S-3017: Hisense 43in Smart TV at Kisumu Mega Plaza for KES 155,996.00
  S-3022: HP 15 Laptop at Nairobi CBD for KES 259,996.00
  S-3036: Tecno Spark 20 at Nakuru Westside for KES 53,996.00
  S-3048: HP 15 Laptop at Kisumu Mega Plaza for KES 129,998.00

High-value sales per store:
  Kisumu Mega Plaza        2
  Nairobi CBD              1
  Nakuru Westside          1
  Westgate                 1
  Mombasa City Mall        0


In [46]:
# Reusable statistics - implement these yourself, no statistics module.
def mean(values: list[float]) -> float:
    """Return the mean of a list of numbers."""
    if not values:
        return 0.0
    return sum(values) / len(values)

print(f"Mean revenue: KES {mean([r['revenue'] for r in rows]):,.2f}")

Mean revenue: KES 28,495.91


In [59]:
def median(values: list[float]) -> float:
    if not values:
        return 0.0
    sorted_values = sorted(values)
    n = len(sorted_values)
    if n % 2 == 0:
        return (sorted_values[n // 2 - 1] + sorted_values[n // 2]) / 2
    else:
        return sorted_values[n // 2]

print(f"Median revenue: KES {median([r['revenue'] for r in rows]):,.2f}")

Median revenue: KES 11,097.00


In [60]:
def minimum_maximum(values: list[float]) -> tuple[float, float]:
    if not values:
        return (0.0, 0.0)
    return (min(values), max(values))
    if values:
        return (min(values), max(values))
        if value in valuues[1:]:
            if value < lowest:
                lowest = value
            if value > highest:
                highest = value
            return (lowest, highest)

    
        
print(f"Minimum revenue: KES {minimum_maximum([r['revenue'] for r in rows])[0]:,.2f}")
print(f"Maximum revenue: KES {minimum_maximum([r['revenue'] for r in rows])[1]:,.2f}")

Minimum revenue: KES 1,499.00
Maximum revenue: KES 259,996.00


In [ ]:
import math
import pandas as pd


def verify_with_pandas(path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print whether each one matches."""
    df = pd.read_csv(path, dtype=str)          

    def to_number(column: str) -> pd.Series:
        text = (
            df[column].astype("string").str.strip()
            .str.replace(r"(?i)^kes\s*", "", regex=True)  
            .str.replace(",", "", regex=False)             
        )
        return pd.to_numeric(text, errors="coerce")        

    df["quantity"] = to_number("quantity")
    df["unit_price_kes"] = to_number("unit_price_kes")
    df = df[(df["quantity"] > 0) & (df["unit_price_kes"] > 0)].copy()  
    df["revenue"] = df["quantity"] * df["unit_price_kes"]

    by_product = df.groupby("product")["revenue"].sum()
    top_product = sorted(by_product.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

    checks = {
        "clean row count":  (my_results["clean_rows"],      len(df)),
        "total revenue":    (my_results["total_revenue"],   float(df["revenue"].sum())),
        "top product":      (my_results["top_product"],     top_product),
        "mean revenue":     (my_results["mean"],            float(df["revenue"].mean())),
        "median revenue":   (my_results["median"],          float(df["revenue"].median())),
        "min revenue":      (my_results["min"],             float(df["revenue"].min())),
        "max revenue":      (my_results["max"],             float(df["revenue"].max())),
        "high-value sales": (my_results["high_value_count"], int((df["revenue"] >= THRESHOLD).sum())),
    }

    print("=== Verification with pandas ===")
    for label, (mine, theirs) in checks.items():
        if isinstance(mine, str) or isinstance(theirs, str):
            ok = mine == theirs
        else:
            ok = math.isclose(mine, theirs, abs_tol=0.01)
        print(f"  {label:<18} mine={mine!r:<22} pandas={theirs!r:<22} {'MATCH' if ok else 'MISMATCH'}")
        

In [ ]:
if __name__ == "__main__":
    # Task 1: read the data
    raw_rows = read_sales(DATA_PATH)
    print()
    
 
    # Task 2: convert and clean
    rows, skipped = convert_rows(raw_rows)
    print(f"\nClean rows: {len(rows)} | Skipped rows: {len(skipped)}")
    for s in skipped:
        print(f"  skipped {s['sale_id']}: {s['reason']}")
 
    # Task 3: total revenue
    total = total_revenue(rows)
    print(f"\nTotal revenue: {KES(total)}")
 
    # Task 4: revenue by product, store and payment method
    by_product = revenue_by(rows, "product")
    by_store = revenue_by(rows, "store")
    by_payment = revenue_by(rows, "payment_method")
 
    print("\nRevenue by product (highest first):")
    for name, amount in sorted(by_product.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {name:<24} {KES(amount):>18}")
    print("\nRevenue by store:")
    for name, amount in sorted(by_store.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {name:<24} {KES(amount):>18}")
    print("\nRevenue by payment method:")
    for name, amount in sorted(by_payment.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {name:<24} {KES(amount):>18}")
 
    # Task 5: top 3 products
    top3 = top_n(by_product, 3)
    print("\nTop 3 products by revenue:")
    for rank, (name, amount) in enumerate(top3, start=1):
        print(f"  {rank}. {name}: {KES(amount)}")
 
    # Task 6: high-value sales
    big = high_value(rows, THRESHOLD)
    print(f"\nHigh-value sales ({KES(THRESHOLD)} or more): {len(big)}")
    for r in big:
        print(f"  {r['sale_id']}: {r['product']} at {r['store']} for {KES(r['revenue'])}")
 
    stores = {r["store"] for r in rows}
    big_by_store = {s: sum(1 for r in big if r["store"] == s) for s in stores}
    print("\nHigh-value sales per store:")
    for name, count in sorted(big_by_store.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {name:<24} {count}")
 
    # Task 7: statistics on per-transaction revenue
    revenues = [r["revenue"] for r in rows]
    low, high = minimum_maximum(revenues)
    avg, mid = mean(revenues), median(revenues)
    print("\nPer-transaction revenue:")
    print(f"  mean    {kes(avg)}")
    print(f"  median  {kes(mid)}")
    print(f"  minimum {kes(low)}")
    print(f"  maximum {kes(high)}")
 
    # Task 8: verify with pandas
    verify_with_pandas(DATA_PATH, {
        "clean_rows": len(rows),
        "total_revenue": total,
        "top_product": top3[0][0],
        "mean": avg,
        "median": mid,
        "min": low,
        "max": high,
        "high_value_count": len(big),
    })

Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


Clean rows: 46 | Skipped rows: 4
  skipped S-3006: quantity 'two' is missing, zero or not numeric
  skipped S-3030: unit price 'N/A' is missing, zero or not numeric
  skipped S-3035: quantity '' is missing, zero or not numeric
  skipped S-3039: unit price '' is missing, zero or not numeric


NameError: name 'kes' is not defined